# ranking_forcast — Predizione per la prossima stagione

La finestra storica viene letta da `configs/config.yaml`: da `first_year` a `last_year`, estremi inclusi. La previsione riguarda la stagione che inizia in `last_year + 1` e comprende tutti i coach la cui cella in quella stagione è diversa da `A`: sono inclusi celle vuote, spazi e risultati già registrati (`W`, `Q`, `N`, `R`). Il codice `A` viene riconosciuto anche in minuscolo o con spazi intorno. Questo permette di fare previsioni anche per stagioni passate. La stagione futura serve solo a selezionare i partecipanti: non entra nei punteggi storici. Posizioni e normalizzazione sono calcolate tra i soli partecipanti selezionati. Se la stagione futura manca o è duplicata, viene segnalato un errore.

Punteggi: **W = 20**, **Q = 6** (secondo o terzo posto, escluso il vincitore), **N = 1**, **A (assente) = 1**, **R = -2**. Questi pesi sono specifici della previsione e indipendenti dalla conversione usata nei grafici dello storico.

- **History score**: somma dei punteggi delle sole stagioni giocate (`W`, `Q`, `N`, `R`), divisa per il numero di partecipazioni. Le assenze non entrano né nella somma né nel denominatore storico. Questa media si applica con almeno 3 partecipazioni. Con 1 o 2 partecipazioni si usa `(W + Q + Q + R + R + R + R + risultati reali + N mancanti) / 10`, aggiungendo rispettivamente 2 o 1 risultati `N`.
- **Last 3 years score**: somma dei punteggi delle ultime tre stagioni della finestra, divisa per 3. Assenze e stagioni precedenti all'inizio della finestra valgono 1.
- **Nessuna partecipazione** nella finestra (anche quando tutte le stagioni sono assenze): lo score recente è `(Q + N + R) / 3 = 5/3`; lo storico è `(W + Q + Q + N + N + N + N + R + R + R) / 10 = 3`.
- **Ranking score**: `(History score + Last 3 years score) / T * 100`, dove T è la somma massima. Le parità usano posizioni come `1, 2, 2, 4`, calcolate prima dell'arrotondamento visuale.

Se T è zero, la normalizzazione non è definita e viene segnalato un errore. Con T negativo la formula inverte l'ordine dei valori normalizzati: la classifica resta ordinata per somma A + B. I punteggi negativi non vengono tagliati.


In [ ]:
from pathlib import Path
from fractions import Fraction
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_results


## Configurazione e dati

In [ ]:
with (PROJECT_ROOT / "configs" / "config.yaml").open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

years = config["year_selection"]
first_year = years["first_year"]
last_year = years["last_year"]
def select_forecast_coaches(data: pd.DataFrame, forecast_year: int) -> list:
    """Seleziona tutti i coach non assenti nella stagione da prevedere."""
    season_years = pd.to_numeric(
        data.iloc[:, 0].astype(str).str.strip().str.split("/").str[0],
        errors="coerce",
    )
    season = data.loc[season_years == forecast_year]
    if len(season) != 1:
        raise ValueError(
            f"Attesa una sola riga per la stagione {forecast_year}: trovate {len(season)}."
        )
    entries = season.iloc[0, 1:]
    return [
        coach for coach, value in entries.items()
        if pd.isna(value) or str(value).strip().upper() != "A"
    ]


data_path = PROJECT_ROOT / config["data"]["file_path"]
raw_data = pd.read_excel(data_path)
forecast_coaches = select_forecast_coaches(raw_data, last_year + 1)
results = load_results(data_path, first_year, last_year)
# Include anche gli esordienti con una colonna interamente vuota nel dataset.
results = results.reindex(columns=forecast_coaches, fill_value="A")

forecast_year = last_year + 1
print(f"Storico: {first_year}/{first_year + 1} – {last_year}/{last_year + 1}")
print(f"Predizione: {forecast_year}/{forecast_year + 1}")
print(f"Coach: {len(results.columns)}")


## Calcolo della classifica

In [ ]:
FORECAST_POINTS = {"W": 20, "Q": 6, "N": 1, "A": 1, "R": -2}


def build_ranking_forcast(results: pd.DataFrame, last_year: int) -> pd.DataFrame:
    """Calcola la previsione sui risultati già selezionati tramite config."""
    recent_years = range(last_year - 2, last_year + 1)
    newcomer_score = Fraction(sum(FORECAST_POINTS[r] for r in ("R", "N", "Q")), 3)
    rows = []
    for coach in results.columns:
        history = results[coach]
        played = history[history != "A"]
        participations = len(played)
        if participations == 0:
            historical = Fraction(
                sum(FORECAST_POINTS[r] for r in "WQQNNNNRRR"), 10
            )
            recent = newcomer_score
        else:
            if participations < 3:
                # Sette risultati di base, risultati reali e N fino a dieci.
                historical_results = list("WQQRRRR") + played.tolist()
                historical_results += ["N"] * (10 - len(historical_results))
                historical = Fraction(
                    sum(FORECAST_POINTS[r] for r in historical_results), 10
                )
            else:
                historical = Fraction(sum(FORECAST_POINTS[r] for r in played), participations)
            recent_results = history.reindex(recent_years, fill_value="A")
            recent = Fraction(sum(FORECAST_POINTS[r] for r in recent_results), 3)
        rows.append({
            "Nome": coach,
            "Partecipazioni": participations,
            "History score": historical,
            "Last 3 years score": recent,
            "Score": historical + recent,
        })

    columns = ["Risultato stimato", "Nome", "Ranking score", "Partecipazioni", "History score", "Last 3 years score", "Score"]
    if not rows:
        return pd.DataFrame(columns=columns)

    # Frazioni esatte: somme matematicamente uguali ricevono la stessa posizione.
    rows.sort(key=lambda row: row["Score"], reverse=True)
    top_score = rows[0]["Score"]
    if top_score == 0:
        raise ValueError("Impossibile normalizzare: la somma massima T è zero.")

    previous_score = None
    position = 0
    for index, row in enumerate(rows, start=1):
        total = row["Score"]
        if total != previous_score:
            position = index
        row["Risultato stimato"] = position
        row["Ranking score"] = float(total / top_score * 100)
        previous_score = total
        for column in ("History score", "Last 3 years score", "Score"):
            row[column] = float(row[column])
    return pd.DataFrame(rows, columns=columns)



In [ ]:
ranking_forcast = build_ranking_forcast(results, last_year)
print(f"ranking_forcast — stagione {forecast_year}/{forecast_year + 1}")
display(ranking_forcast.style.hide(axis="index").format({
    "History score": "{:.3f}",
    "Last 3 years score": "{:.3f}",
    "Score": "{:.3f}",
    "Ranking score": "{:.2f}",
}))


## Verifica delle regole
Esegui questi esempi per controllare i casi particolari.

In [ ]:
# Esempi sintetici: assenze, storico, finestra breve, esordienti e parità.
from math import isclose

sample = pd.DataFrame({
    "Vincitore": ["W", "W", "W", "W"],
    "Pippo": ["W", "A", "N", "Q"],
    "Pluto": ["W", "A", "N", "Q"],
    "Topolino": ["A", "A", "A", "A"],
}, index=[2022, 2023, 2024, 2025])
check = build_ranking_forcast(sample, 2025).set_index("Nome")
assert check["Risultato stimato"].tolist() == [1, 2, 2, 4]
assert check.loc["Vincitore", "Ranking score"] == 100
assert check.loc["Pippo", "History score"] == 9
assert isclose(check.loc["Pippo", "Last 3 years score"], 8 / 3)
assert isclose(check.loc["Topolino", "History score"], 3)
assert isclose(check.loc["Topolino", "Last 3 years score"], 5 / 3)

for values, expected in [(["W"], 22 / 3), (["W", "Q"], 27 / 3)]:
    short = pd.DataFrame({"Coach": values}, index=range(2026 - len(values), 2026))
    assert isclose(build_ranking_forcast(short, 2025).iloc[0]["Last 3 years score"], expected)

empty_history = pd.DataFrame(columns=["Esordiente"], index=pd.Index([], dtype=int))
newcomer = build_ranking_forcast(empty_history, 2025).iloc[0]
assert isclose(newcomer["History score"], 3)
assert isclose(newcomer["Last 3 years score"], 5 / 3)

relegated = pd.DataFrame({"Coach": ["R"]}, index=[2025])
assert build_ranking_forcast(relegated, 2025).iloc[0]["History score"] == 2.4
zero = pd.DataFrame({"Coach": ["R", "N", "N"]}, index=[2023, 2024, 2025])
try:
    build_ranking_forcast(zero, 2025)
except ValueError as error:
    assert "T è zero" in str(error)
else:
    raise AssertionError("T = 0 deve segnalare normalizzazione non definita")
print("Verifiche superate.")

# La selezione usa esclusivamente la riga futura e conserva i nuovi coach.
selection_sample = pd.DataFrame({
    "Stagione": ["2025/26", "2026/27"],
    "Assente": ["W", "A"],
    "Vuoto": ["N", None],
    "Spazi": ["Q", "  "],
    "Esordiente": [None, None],
    "Con esito": ["N", "W"],
})
selected = select_forecast_coaches(selection_sample, 2026)
assert selected == ["Vuoto", "Spazi", "Esordiente", "Con esito"]
selected_history = pd.DataFrame({
    "Assente": ["W"], "Vuoto": ["N"], "Spazi": ["Q"], "Con esito": ["N"],
}, index=[2025]).reindex(columns=selected, fill_value="A")
selected_ranking = build_ranking_forcast(selected_history, 2025)
assert set(selected_ranking["Nome"]) == set(selected)
assert selected_ranking.iloc[0]["Ranking score"] == 100
assert selected_ranking["Risultato stimato"].tolist() == [1, 2, 3, 3]
assert select_forecast_coaches(selection_sample.fillna("A").replace("  ", "A"), 2026) == ["Con esito"]
assert build_ranking_forcast(selected_history.iloc[:, :0], 2025).empty
for invalid_data in (selection_sample.iloc[:1], pd.concat([selection_sample, selection_sample])):
    try:
        select_forecast_coaches(invalid_data, 2026)
    except ValueError:
        pass
    else:
        raise AssertionError("La stagione futura deve essere presente e unica")
print("Verifiche selezione partecipanti superate.")

# Uno o due risultati reali: tutte le combinazioni, anche fuori dagli ultimi 3 anni.
from itertools import product
for count in (1, 2):
    for actual in product("WQNR", repeat=count):
        expected_history = (24 + sum(FORECAST_POINTS[r] for r in actual) + 3 - count) / 10
        for start in (2020, 2024):
            sparse = pd.DataFrame({"Coach": list(actual)}, index=range(start, start + count))
            row = build_ranking_forcast(sparse, 2025).iloc[0]
            assert row["Partecipazioni"] == count
            assert isclose(row["History score"], expected_history)
            expected_recent = sum(FORECAST_POINTS[sparse["Coach"].get(year, "A")] for year in (2023, 2024, 2025)) / 3
            assert isclose(row["Last 3 years score"], expected_recent)
            assert isclose(row["Score"], expected_history + expected_recent)
print("Verifiche storico con 1 o 2 partecipazioni superate.")

# Previsioni nel passato: ogni risultato registrato indica una partecipazione.
for result in ("W", "Q", "N", "R", "", "  ", None):
    historical_target = pd.DataFrame({"Stagione": ["2024/25", "2025/26", "2026/27"],
                                      "Coach": ["N", result, "A"]})
    assert select_forecast_coaches(historical_target, 2025) == ["Coach"]
for absent in ("A", " a "):
    historical_target.loc[1, "Coach"] = absent
    assert select_forecast_coaches(historical_target, 2025) == []
print("Verifiche previsione nel passato superate.")
